<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## Feasible Delivery Tours

**Define a complete delivery tour before comparing travel distances.**

Keep the five-city symmetric distance matrix, the requirement to visit every city once and return, and the total-distance objective. The decision is a discrete visit order. The clock observations, calibration equality, and correction bound do not apply to this delivery system. The new formulation replaces continuous clock settings with the order of five visits.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 1 · Choose a complete tour

A vehicle must visit cities $0,1,2,3,4$ once each and return to its starting city. A **tour** is that complete closed route. Shorter total distance means less travel.

Let $C_{ij}$ be the fixed travel distance from city $i$ to city $j$. The source gives no physical unit, so we report **distance units**. All city pairs are connected, and the distances are symmetric: $C_{ij}=C_{ji}$. The numbers are travel costs; we do not assume they are straight-line distances on a map.

| From / to | $0$ | $1$ | $2$ | $3$ | $4$ |
|:---|---:|---:|---:|---:|---:|
| $0$ | 0 | 2 | 9 | 10 | 7 |
| $1$ | 2 | 0 | 8 | 5 | 6 |
| $2$ | 9 | 8 | 0 | 4 | 3 |
| $3$ | 10 | 5 | 4 | 0 | 1 |
| $4$ | 7 | 6 | 3 | 1 | 0 |

Write the decision as $x=(v_0,v_1,v_2,v_3,v_4,v_5)$, where $v_k$ is the city visited at position $k$. Feasibility requires that $v_0,\ldots,v_4$ contain every city exactly once and $v_5=v_0$. The objective is

$$
f(x)=\sum_{k=0}^{4} C_{v_k,v_{k+1}}.
$$

The final term includes the return trip. For example,

$$
f(2,4,3,1,0,2)=3+1+5+2+9=20.
$$

The distance matrix is fixed data. The visit order is the physical decision. Here the raw performance output, total distance, is also the scalar objective. A route with a missing city or return trip is rejected before any score comparison.


In [ ]:
import numpy as np

# Fixed travel distances, copied from the supplied example_1.py
DISTANCES = np.array([
    [0, 2, 9, 10, 7],
    [2, 0, 8, 5, 6],
    [9, 8, 0, 4, 3],
    [10, 5, 4, 0, 1],
    [7, 6, 3, 1, 0],
], dtype=int)
CITY_COUNT = len(DISTANCES)


def evaluate_tour(tour):
    tour = tuple(tour)
    valid_labels = all(isinstance(city, (int, np.integer))
                       and 0 <= city < CITY_COUNT for city in tour)
    feasible = (len(tour) == CITY_COUNT + 1 and valid_labels
                and tour[0] == tour[-1]
                and set(tour[:-1]) == set(range(CITY_COUNT)))
    if not feasible:
        return {"decision": tour, "feasible": False,
                "edge_distances": None, "objective": None}
    edges = np.array([DISTANCES[a, b] for a, b in zip(tour, tour[1:])])
    return {"decision": tour, "feasible": True,
            "edge_distances": edges, "objective": int(edges.sum())}


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 2 · Include the return trip

The open route $(2,4,3,1,0)$ has distance $3+1+5+2=11$, but it omits the required return trip. Adding the edge from 0 to 2 gives $11+9=20$. The incomplete route is rejected before any score comparison.

The rows below keep the same visit order. The orange return edge closes the lower route and adds 9 distance units. The edge labels give travel distances; line lengths do not represent those distances.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/05-1_heuristic_optimization_greedy_and_local_search/assets/complete_tour_return.png" alt="Both rows visit cities 2, 4, 3, 1, and 0 in that order, using edges of distance 3, 1, 5, and 2. The open route has distance 11 and is incomplete. The closed tour adds the return edge from 0 to 2, of distance 9, for a feasible total of 20." width="512" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: #fff;">
</div>


In [ ]:
for tour in ((2, 4, 3, 1, 0, 2), (2, 4, 3, 1, 0)):
    result = evaluate_tour(tour)
    if not result["feasible"]:
        print(tour, "rejected: visit every city once and return to the start")
        continue
    print(tour, "edge distances:", result["edge_distances"],
          "total:", result["objective"], "distance units")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 3 · Source material and method reference

The supplied `Lecture_10_1_Heuristic_Optimization/10_1.pptx` introduces heuristic construction rules. Its `ExampleCode_10_1/example_1.py` supplies the five-city distance matrix, nearest-neighbor rule, and original start at city 2. This notebook states feasibility and calculates the complete tour distance.

Michael A. Trick's [Heuristics](https://mat.tepper.cmu.edu/orclass/integer/node17.html) describes nearest-neighbor tour construction and two-opt tour improvement. The worked calculations use the supplied matrix.
